# 01 - Build the training data

Runs the data pipeline on Colab: **download -> clean -> LaBSE filter -> train tokenizer**. The output (line-aligned `*.hi`/`*.en` splits + a SentencePiece model) is what `02_train` trains on.

**Before running:** set a **GPU runtime** (Runtime -> Change runtime type -> GPU) - the LaBSE step needs it.

**Persistence:** Colab's `/content` is wiped on reset, and the LaBSE pass is slow, so for repeat sessions point `DATA_ROOT` (config cell) at a mounted Google Drive folder. Run the cells top to bottom.

In [2]:
!git add notebooks/01_data.ipynb notebooks/00_run_tests.ipynb && git commit -m "Add 01_data driver; harden test notebook" && git push

fatal: not a git repository (or any of the parent directories): .git


## 1. Repo + dependencies

In [3]:
import os, sys

# If auto-detect fails, set this to your repo path, e.g. "/content/hindi-translator".
REPO_DIR = None

d = REPO_DIR or os.getcwd()
while d != "/" and not os.path.exists(os.path.join(d, "pyproject.toml")):
    d = os.path.dirname(d)
assert os.path.exists(os.path.join(d, "pyproject.toml")), "repo not found - set REPO_DIR above"
os.chdir(d)
sys.path.insert(0, os.path.abspath("src"))  # import nmt straight from src/ (robust to editable-install state)
print("repo root:", d)

AssertionError: repo not found - set REPO_DIR above

In [ ]:
# package deps (datasets, sentence-transformers, indic-nlp-library, sentencepiece, ...) + fasttext for langid
!pip install -e . fasttext-wheel

In [ ]:
# fastText language-id model used by clean.py (~126 MB, one-time; -nc skips if already present)
LID_PATH = "lid.176.bin"
!wget -nc -q https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
print("langid model present:", os.path.exists(LID_PATH))

## 2. Config (paths)

In [ ]:
from nmt.config import DataConfig

# Where data + tokenizer get written. /content is wiped on runtime reset; for persistence across
# sessions, mount Drive and point DATA_ROOT at a Drive folder:
#   from google.colab import drive; drive.mount('/content/drive')
#   DATA_ROOT = '/content/drive/MyDrive/hindi-translator-data'
DATA_ROOT = "data"  # repo-local (data/raw, data/cache) - fine for a first run

cfg = DataConfig(raw_dir=f"{DATA_ROOT}/raw", cache_dir=f"{DATA_ROOT}/cache")
print("raw_dir:", cfg.raw_dir, "| cache_dir:", cfg.cache_dir, "| vocab:", cfg.vocab_size, "| corpus:", cfg.corpus)

## 3. Download IITB

Pulls the IIT Bombay Hi-En corpus from HuggingFace and writes line-aligned `train`/`dev`/`test` `.hi`/`.en` files to `raw_dir`. Idempotent - skips splits already staged.

In [ ]:
from nmt.data.download import download

raw_paths = download(cfg)            # {split: (hi_path, en_path)}
print("sample hi:", open(raw_paths["train"][0], encoding="utf-8").readline().strip())
print("sample en:", open(raw_paths["train"][1], encoding="utf-8").readline().strip())

## 4. Clean

NFC + Indic normalization, fastText language-id filtering, length-ratio gate, dedupe, and removal of train pairs that leak into dev/test. **Only train is filtered**; dev/test are normalized but never dropped. Prints the kept/dropped breakdown.

In [ ]:
from nmt.data.clean import clean

clean_paths = clean(cfg, lid_path=LID_PATH)   # writes *.clean.hi/.en; prints kept/dropped stats

## 5. LaBSE filter (GPU; slow, one-time, cached)

Embeds both sides of each cleaned train pair with LaBSE and drops pairs whose cross-lingual cosine similarity is below `cfg.labse_threshold` (0.70), removing misaligned/noisy pairs. This is the long step - cached as `train.labse.*`. LaBSE is a pretrained *filter*, not part of the from-scratch model.

In [ ]:
from nmt.data.labse_filter import labse_filter

labse_paths = labse_filter(cfg)     # GPU auto-used if present; writes train.labse.*; dev/test pass through

## 6. Train the tokenizer

One shared SentencePiece **unigram 16k** vocab over both languages on the LaBSE-filtered train, with `byte_fallback` (no `<unk>`) and the fixed id contract pad/unk/bos/eos = 0/1/2/3. Writes `spm_unigram_16000.model`.

In [ ]:
from nmt.data.tokenizer import train_tokenizer

tok = train_tokenizer(cfg, labse_paths["train"][0], labse_paths["train"][1])
print("vocab_size:", tok.vocab_size, "| pad/unk/bos/eos:", tok.pad_id, tok.unk_id, tok.bos_id, tok.eos_id)

## 7. Sanity checks

In [ ]:
# line counts (hi/en must match per split) + a tokenizer round-trip
def _count(p):
    with open(p, encoding="utf-8") as f:
        return sum(1 for _ in f)

for split, (h, e) in labse_paths.items():
    nh, ne = _count(h), _count(e)
    print(f"{split:5s}: {nh} hi / {ne} en  {'OK' if nh == ne else 'MISMATCH'}")

sample = open(labse_paths["train"][0], encoding="utf-8").readline().strip()
ids = tok.encode(sample, add_eos=True)
print("\nsample :", sample)
print("ids    :", ids[:20], "...")
print("decoded:", tok.decode(ids))
print("has <unk>?", tok.unk_id in ids, "(should be False - byte_fallback)")